In [13]:
import os
import boto3
import re
from botocore.client import Config

# --- CONFIGURACIÓN ---
endpoint = "http://minio-service.t-moviles-ai.svc.cluster.local:9000"
creds = {"key": "minio", "secret": "minio123"}
bucket_name = "modelos-prod"
local_folder = "./modelo_t-moviles_onnx"

# --- INICIALIZACIÓN ---
s3 = boto3.client('s3', endpoint_url=endpoint, 
                  aws_access_key_id=creds["key"], 
                  aws_secret_access_key=creds["secret"],
                  config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
                  region_name='us-east-1')

def get_latest_version_number(bucket):
    """Escanea el bucket buscando carpetas con formato 'vX'"""
    try:
        # Listamos los 'directorios' raíz
        response = s3.list_objects_v2(Bucket=bucket, Delimiter='/')
        prefixes = response.get('CommonPrefixes', [])
        
        versions = [0] # Base por si el bucket está vacío
        for p in prefixes:
            folder = p.get('Prefix').strip('/')
            # Extraemos el número de carpetas tipo 'v1', 'v2', etc.
            match = re.match(r'^v(\d+)$', folder)
            if match:
                versions.append(int(match.group(1)))
        
        return max(versions)
    except Exception as e:
        print(f"⚠️ No se pudo leer versiones previas (asumiendo v0): {e}")
        return 0

def upload_incrementing_version():
    # 1. Validar archivos locales
    if not os.path.exists(local_folder) or not os.listdir(local_folder):
        print(f"❌ Error: No hay archivos en {local_folder} para subir.")
        return

    # 2. Calcular nueva versión
    current_max = get_latest_version_number(bucket_name)
    new_version = f"v{current_max + 1}"
    
    print(f"🔍 Última versión detectada: v{current_max}")
    print(f"🚀 Subiendo nueva versión: {new_version}...")

    # 3. Subida de archivos
    for file in os.listdir(local_folder):
        local_path = os.path.join(local_folder, file)
        if os.path.isfile(local_path):
            s3_key = f"{new_version}/{file}"
            s3.upload_file(local_path, bucket_name, s3_key)
            print(f"   ✅ {s3_key}")

    print(f"\n✨ ¡Listo! Modelo T-Moviles Argentina {new_version} desplegado en MinIO.")
    return new_version

# Ejecutar
version_desplegada = upload_incrementing_version()

🔍 Última versión detectada: v1
🚀 Subiendo nueva versión: v2...
   ✅ v2/config.json
   ✅ v2/tokenizer.json
   ✅ v2/model.onnx
   ✅ v2/vocab.txt
   ✅ v2/special_tokens_map.json
   ✅ v2/tokenizer_config.json

✨ ¡Listo! Modelo Movistar Argentina v2 desplegado en MinIO.
